<a href="https://colab.research.google.com/github/adhikarisamundra/Aggression-Game-Analysis/blob/main/Grid_Map_Aggression_Game_Solver.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import sys
import time

# --- Global Cache for Memoization ---
# Stores results of states we've already analyzed to avoid re-calculating.
memoization_cache = {}

# --- Core Game Logic & Solver ---

def evaluate_final_state(grid, p1_troops, p2_troops, initiative_winner_known):
    """
    Evaluates a terminal game state by fully simulating the end-game from the base code.
    Returns a score: +1 for P1 win, -1 for P2 win, according to the EXACT rules.
    """

    # --- Simulate End Game Based on Provided Simulator Rules ---
    final_grid = [row[:] for row in grid]

    # 1. Determine Initiative Winner (Rule from the Simulator)
    # If a player ran out of troops, they get initiative.
    if initiative_winner_known:
        first_attacker = initiative_winner_known
    # If board filled up, P1 gets initiative by default.
    else:
        first_attacker = 'P1'

    second_attacker = 'P2' if first_attacker == 'P1' else 'P1'

    # 2. Simulate the two attack phases
    final_grid = run_simulated_attack(final_grid, first_attacker)
    final_grid = run_simulated_attack(final_grid, second_attacker)

    # 3. Calculate final winner using the three-tiered victory conditions
    p1_cells, p2_cells = 0, 0
    final_p1_troops, final_p2_troops = 0, 0

    for r in range(len(final_grid)):
        for c in range(len(final_grid[0])):
            if final_grid[r][c]:
                player, troops = final_grid[r][c]
                if player == 'P1':
                    p1_cells += 1
                    final_p1_troops += troops
                else:
                    p2_cells += 1
                    final_p2_troops += troops

    # Tier 1: Area Control
    if p1_cells > p2_cells: return 1
    if p2_cells > p1_cells: return -1

    # Tier 2: Troop Count
    if final_p1_troops > final_p2_troops: return 1
    if final_p2_troops > final_p1_troops: return -1

    # Tier 3: Initiative
    return 1 if first_attacker == 'P1' else -1


def run_simulated_attack(grid, attacking_player):
    """A non-printing version of the run_attack_phase function from the simulator."""
    if not attacking_player:
        return grid

    defending_player = 'P2' if attacking_player == 'P1' else 'P1'
    rows, cols = len(grid), len(grid[0])
    cells_to_remove = []

    for r in range(rows):
        for c in range(cols):
            if grid[r][c] and grid[r][c][0] == defending_player:
                defender_troops = grid[r][c][1]
                total_attacker_strength = 0

                # Check all 8 neighbors for attackers
                for dr in [-1, 0, 1]:
                    for dc in [-1, 0, 1]:
                        if dr == 0 and dc == 0: continue

                        nr, nc = r + dr, c + dc

                        if 0 <= nr < rows and 0 <= nc < cols and grid[nr][nc] and grid[nr][nc][0] == attacking_player:
                            # Replicate the 3x3 center-attack rule from the simulator
                            is_3x3_grid = (rows == 3 and cols == 3)
                            attacker_is_center = (nr == 1 and nc == 1)
                            is_diagonal_attack = (dr != 0 and dc != 0)

                            if is_3x3_grid and attacker_is_center and is_diagonal_attack:
                                continue

                            total_attacker_strength += grid[nr][nc][1]

                if total_attacker_strength > defender_troops:
                    cells_to_remove.append((r, c))

    if not cells_to_remove:
        return grid # Return original grid if no changes were made

    new_grid = [row[:] for row in grid]
    for r_rem, c_rem in cells_to_remove:
        new_grid[r_rem][c_rem] = None
    return new_grid


def get_possible_moves(grid, turn_player, p1_troops, p2_troops):
    """Generates all valid moves for the current player."""
    moves = []
    current_troops = p1_troops if turn_player == 'P1' else p2_troops
    if current_troops == 0:
        return []

    # Heuristic to prune the number of troop placements to check
    troops_to_place_options = {1}
    if current_troops > 1:
        troops_to_place_options.add(current_troops)
        if current_troops <= 6:
             for i in range(2, current_troops):
                troops_to_place_options.add(i)
        else:
            troops_to_place_options.add(current_troops // 2)

    for r in range(len(grid)):
        for c in range(len(grid[0])):
            if grid[r][c] is None:
                for k in troops_to_place_options:
                    if k <= current_troops:
                        moves.append({'k': k, 'pos': (r, c)})
    return moves


def alpha_beta_solver(grid, p1_troops, p2_troops, turn_player, alpha, beta, initiative_winner):
    """High-efficiency solver with Alpha-Beta Pruning and Heuristic Move Ordering."""

    board_full = all(cell is not None for row in grid for cell in row)
    if (p1_troops == 0 and p2_troops == 0) or board_full:
        return evaluate_final_state(grid, p1_troops, p2_troops, initiative_winner)

    canonical_key = (tuple(map(tuple, grid)), p1_troops, p2_troops, turn_player)
    if canonical_key in memoization_cache:
        return memoization_cache[canonical_key]

    possible_moves = get_possible_moves(grid, turn_player, p1_troops, p2_troops)

    # --- HEURISTIC MOVE ORDERING ---
    # Sorts moves to check ones with fewer troops first. This makes alpha-beta
    # pruning more effective by analyzing "smarter" moves earlier.
    possible_moves.sort(key=lambda move: move['k'])

    if not possible_moves:
        return alpha_beta_solver(
            grid, p1_troops, p2_troops,
            'P2' if turn_player == 'P1' else 'P1',
            alpha, beta, initiative_winner
        )

    if turn_player == 'P1': # Maximizing player
        best_value = -float('inf')
        for move in possible_moves:
            r, c = move['pos']
            k = move['k']

            # Make move
            grid[r][c] = ('P1', k)
            new_p1_troops = p1_troops - k
            new_initiative_winner = initiative_winner
            if new_p1_troops == 0 and not initiative_winner:
                new_initiative_winner = 'P1'

            value = alpha_beta_solver(grid, new_p1_troops, p2_troops, 'P2', alpha, beta, new_initiative_winner)
            best_value = max(best_value, value)
            alpha = max(alpha, best_value)

            # Unmake move (backtrack)
            grid[r][c] = None

            if beta <= alpha:
                break # Prune

        memoization_cache[canonical_key] = best_value
        return best_value

    else: # Minimizing Player (P2)
        best_value = float('inf')
        for move in possible_moves:
            r, c = move['pos']
            k = move['k']

            # Make move
            grid[r][c] = ('P2', k)
            new_p2_troops = p2_troops - k
            new_initiative_winner = initiative_winner
            if new_p2_troops == 0 and not initiative_winner:
                new_initiative_winner = 'P2'

            value = alpha_beta_solver(grid, p1_troops, new_p2_troops, 'P1', alpha, beta, new_initiative_winner)
            best_value = min(best_value, value)
            beta = min(beta, best_value)

            # Unmake move
            grid[r][c] = None

            if beta <= alpha:
                break # Prune

        memoization_cache[canonical_key] = best_value
        return best_value


# --- Main Execution ---

def get_validated_input(prompt, max_val=5):
    """Helper function to get and validate user input."""
    while True:
        try:
            val = int(input(prompt))
            if 1 <= val <= max_val: return val
            else: print(f"Error: Value must be between 1 and {max_val}.")
        except (ValueError, EOFError):
            print("\nInvalid input. Exiting.")
            sys.exit(1)

def solve_single_game():
    """Sets up and runs the enhanced game solver."""
    print("--- Aggression: Optimal Game Solver ---")

    try:
        # NOTE: Solver is limited to smaller grids due to extreme complexity.
        rows = get_validated_input("Enter grid rows (m, 1-5): ")
        cols = get_validated_input("Enter grid columns (n, 1-5): ")

        if rows * cols > 9:
            print("\n\033[93mWarning: Analysis for this grid size may be very slow.\033[0m")
            consent = input("Do you wish to proceed? (yes/no): ").lower()
            if not consent.startswith('y'):
                print("Analysis cancelled by user.")
                return

        global memoization_cache
        memoization_cache.clear()

        initial_troops = rows * cols
        initial_grid = [[None for _ in range(cols)] for _ in range(rows)]

        print("\nAnalyzing game tree... This may take a moment.")
        start_time = time.time()

        final_score = alpha_beta_solver(
            initial_grid, initial_troops, initial_troops, 'P1',
            -float('inf'), float('inf'), None
        )

        end_time = time.time()
        winner = "Player 1" if final_score == 1 else "Player 2"

        print("\n--- ANALYSIS COMPLETE ---")
        print(f"Grid Size: {rows}x{cols}")
        print(f"Unique States Analyzed: {len(memoization_cache)}")
        print(f"Time Taken: {end_time - start_time:.4f} seconds")
        print("-" * 25)
        print(f"\033[1mResult: {winner} has a guaranteed win under optimal play.\033[0m")

    except Exception as e:
        print(f"\nAn unexpected error occurred: {e}")
        import traceback
        traceback.print_exc()

if __name__ == "__main__":
    solve_single_game()
